# Phase A — Neuron-side: sinh `bridge_scales.json` trên Colab GPU

**Mục tiêu:** sửa nhóm neuron biểu hiện tính trạng (PD models), đo DN spike rates,
tính `motor_scale` / `coupling_scale`, ghi `<model>_bridge_scales.json`
để kéo về chạy body-side (`run_brain_driven_experiment.py`).

```
Sensory stimulus (p9) -> fly-brain LIF (138k neurons, GPU)
  -> DN spike readout (P9 forward / DNa01-02 turn)
  -> rate ratio vs HEALTHY baseline
  -> motor_scale, coupling_scale -> bridge_scales.json
```

**Ranh giới khoa học:** mọi scale là tỷ lệ spike-rate tính toán giữa điều kiện
model và healthy — KHÔNG phải nồng độ dopamine hay mức thoái hóa neuron thật.

**Cần:** Runtime > Change runtime type > **GPU (T4)**.

In [ ]:
# @title 1. Kiểm tra môi trường GPU
import torch, sys
print('Python :', sys.version.split()[0])
print('torch  :', torch.__version__)
assert torch.cuda.is_available(), (
    'KHONG co GPU! Vao Runtime > Change runtime type > chon GPU (T4) roi chay lai.')
print('CUDA   :', torch.cuda.get_device_name(0))

In [ ]:
# @title 2. Clone fly-brain + tải file lớn Git LFS (~180 MB)
import os, pathlib
REPO_DIR = pathlib.Path('/content/fly-brain')
if not (REPO_DIR / '.git').exists():
    !git lfs install --skip-repo
    !git clone https://github.com/erojasoficial-byte/fly-brain.git {REPO_DIR}
    !cd {REPO_DIR} && git lfs pull
else:
    print('fly-brain da ton tai, bo qua clone.')

In [ ]:
# @title 3. Xác minh dữ liệu connectome đã về đầy đủ
import os
EXPECTED = {
    '2025_Completeness_783.csv':  3_465_987,
    '2025_Connectivity_783.parquet': 100_804_642,
    'plastic_weights.pt':        60_369_156,
}
data_dir = REPO_DIR / 'data'
ok = True
for name, size in EXPECTED.items():
    p = data_dir / name
    actual = p.stat().st_size if p.exists() else -1
    status = 'OK ' if actual == size else ('LFS_POINTER!' if actual < 1000 else 'SIZE_DIFF')
    ok &= (status == 'OK ')
    print(f'{status:12} {name:32} {actual:>12,} bytes')
assert ok, ('File loi — thuong do thieu `git lfs pull`. Chay lai cell 2.')
print('Du lieu day du.')

In [ ]:
# @title 4. Import brain_body_bridge từ repo
import sys
sys.path.insert(0, str(REPO_DIR))
import brain_body_bridge as bbb
print('Import OK. DT =', bbb.DT, 'ms | DN groups:', list(bbb.DN_GROUPS.keys()))

## Giao thức đo và định nghĩa "sửa neuron biểu hiện tính trạng"

* Script gốc sinh bridge_scales không được công bố trong repo, nên notebook này
  cài đặt giao thức minh bạch, tái lập được bằng đúng public API:
  * **Healthy**: não nguyên trạng, kích thích `p9` (đi tới, 100 Hz).
  * **PD model**: im lặng (silence = zero toàn bộ synapse vào/ra) một **tỷ lệ**
    neuron dopaminergic (DAN) chọn theo thứ tự bảng chú giải FlyWire.
  * **age25** = mức tổn thương mạnh hơn (fraction cao hơn); `parkin_OE` dùng
    cùng mức trong giao thức rút gọn này (dataset ghi nhận 2 điều kiện này
    có cùng bộ scale).
  * Đo sau warm-up để loại transient, dùng cửa sổ trung bình của decoder.
* `motor_scale`   = forward-DN rate(model) / forward-DN rate(healthy)
* `coupling_scale`= turn-DN rate(model)     / turn-DN rate(healthy)

In [ ]:
# @title 5. Cấu hình thí nghiệm (chỉnh tại đây nếu muốn)
STIMULUS      = 'p9'      # forward-walking drive
WARMUP_MS     = 300.0     # bỏ qua giai đoạn khởi động
MEASURE_MS    = 700.0     # cửa sổ đo chính
DT_MS         = bbb.DT    # 0.1 ms/step
WARMUP_STEPS  = int(WARMUP_MS / DT_MS)
MEASURE_STEPS = int(MEASURE_MS / DT_MS)

# fraction (%) neuron DAN bị im lặng cho từng model
# (tham số hóa theo hướng dataset: pink1 tăng nhẹ, age25 giảm mạnh, v.v.)
MODEL_SPECS = {
    'pink1':                 {'dan_fraction': 0.15},
    'parkin':                {'dan_fraction': 0.45},
    'dj1':                   {'dan_fraction': 0.30},
    'lrrk2':                 {'dan_fraction': 0.10},
    'complexI':              {'dan_fraction': 0.20},
    'pink1_age25':           {'dan_fraction': 0.50},
    'pink1_parkin_OE_age25': {'dan_fraction': 0.50},
}
MODELS = list(MODEL_SPECS)
print('Models:', MODELS)

In [ ]:
# @title 6. Xác định quần thể neuron dopaminergic (DAN) từ annotations
import pandas as pd
ann = pd.read_csv(REPO_DIR / 'data' / 'flywire_annotations.tsv', sep='\t', low_memory=False)
ann.columns = [c.strip().lower() for c in ann.columns]

id_col = next(c for c in ann.columns if 'root' in c and 'id' in c)
type_col = next((c for c in ('celltype', 'cell_type', 'type') if c in ann.columns), None)
nt_col = next((c for c in ('nt', 'neurotransmitter', 'predicted_nt') if c in ann.columns), None)
print(f'columns used: id={id_col}, type={type_col}, nt={nt_col}')

mask = pd.Series(False, index=ann.index)
if type_col:
    mask |= ann[type_col].astype(str).str.upper().str.startswith('DAN')
if nt_col:
    mask |= ann[nt_col].astype(str).str.upper().eq('DA')
dan_ids = ann.loc[mask, id_col].dropna().unique().tolist()
print(f'Dopaminergic neurons found: {len(dan_ids)}')

In [ ]:
# @title 7. Khởi tạo BrainEngine (lần đầu ~vài phút: nạp connectome lên GPU)
engine = bbb.BrainEngine(device='cuda')
decoder = bbb.DNRateDecoder(window_ms=50.0, dt_ms=DT_MS, max_rate=200.0)
for name in ('forward', 'turn_L', 'turn_R'):
    decoder.register_population(name)
    engine.register_population(name,
        [engine.dn_indices[n] for n in bbb.DN_GROUPS[name] if n in engine.dn_indices])

In [ ]:
# @title 8. Hàm đo và hàm im lặng neuron
import numpy as np, torch

def silence_neurons(indices):
    """Zero moi synapse vao/ra cua cac neuron chi dinh (theo README fly-brain)."""
    idx = torch.as_tensor(sorted(set(int(i) for i in indices)),
                          device=engine.device, dtype=torch.int64)
    vals = engine._syn_vals
    cols = engine._col_idx
    posts = engine._post_idx
    kill_in  = torch.isin(cols, idx)
    kill_out = torch.isin(posts, idx)
    vals[kill_in | kill_out] = 0.0
    engine.rates[0, idx] = 0.0
    return int((kill_in | kill_out).sum())

def run_condition(silenced_indices=()):
    """Chay 1 dieu kien: reset trang thai, kich thich p9, do DN rates."""
    engine.state = engine.model.state_init()
    engine._spike_acc.zero_()
    engine._hebb_count = 0
    engine.set_stimulus(STIMULUS)
    if silenced_indices:
        engine.rates[0, torch.as_tensor(sorted(silenced_indices),
                       device=engine.device)] = 0.0
    decoder.__init__(window_ms=50.0, dt_ms=DT_MS, max_rate=200.0)
    for name in ('forward', 'turn_L', 'turn_R'):
        decoder.register_population(name)
    for step in range(WARMUP_STEPS + MEASURE_STEPS):
        spk = engine.step()
        dn = {n: spk[0, i].item() for n, i in engine.dn_indices.items()}
        pops = {k: spk[0, v].sum().item() for k, v in engine.populations.items()}
        decoder.update(dn, pops)
    fwd = np.mean([decoder.get_group_rate('forward')])
    turn = np.mean([decoder.get_pop_rate('turn_L'), decoder.get_pop_rate('turn_R')])
    return {'forward_rate_hz': float(fwd), 'turn_rate_hz': float(turn)}

In [ ]:
# @title 9. Đo HEALTHY baseline (mốc chuẩn)
healthy = run_condition()
print('HEALTHY:', healthy)
assert healthy['forward_rate_hz'] > 0, 'Forward DN khong phat — kiem tra stimulus/DN mapping.'

In [ ]:
# @title 10. Sửa neuron theo từng model → đo → sinh bridge_scales.json
import json, pathlib, datetime

out_dir = REPO_DIR / 'data' / 'results' / 'pd'
out_dir.mkdir(parents=True, exist_ok=True)

results = {}
for model in MODELS:
    frac = MODEL_SPECS[model]['dan_fraction']
    k = max(1, int(round(len(dan_ids) * frac)))
    targets = dan_ids[:k]
    tensor_idx = [engine.flyid2i[i] for i in targets if i in engine.flyid2i]

    # nap lai trong luong goc truoc khi silence (moi model doc lap)
    engine._syn_vals.copy_(engine._abs_orig * engine._sign_mask)
    n_syn = silence_neurons(tensor_idx)

    m = run_condition(engine.flyid2i[i] for i in targets if i in engine.flyid2i)
    scales = {
        'motor_scale':    round(m['forward_rate_hz'] / healthy['forward_rate_hz'], 6),
        'coupling_scale': round(m['turn_rate_hz']   / healthy['turn_rate_hz'], 6),
        'model':          model,
        'protocol': {
            'stimulus': STIMULUS,
            'dan_silenced': len(tensor_idx),
            'dan_total': len(dan_ids),
            'dan_fraction': frac,
            'synapses_zeroed': n_syn,
            'warmup_ms': WARMUP_MS, 'measure_ms': MEASURE_MS,
        },
        'rates': m, 'healthy_rates': healthy,
        'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
        'note': ('Computational DN-rate ratios, not dopamine concentration '
                 'or biological neuron-loss measurements.'),
    }
    path = out_dir / f'{model}_bridge_scales.json'
    path.write_text(json.dumps(scales, indent=2) + '\n', encoding='utf-8')
    results[model] = scales
    print(f"{model:24} motor={scales['motor_scale']:.6f}  "
          f"coupling={scales['coupling_scale']:.6f}  -> {path.name}")

print('\nXong. File nam tai:', out_dir)

In [ ]:
# @title 11. Đối chiếu với dataset tham chiếu (frozen evidence)
REFERENCE = {
    'pink1':                {'motor_scale': 1.148515, 'coupling_scale': 1.047619},
    'pink1_age25':          {'motor_scale': 0.782609, 'coupling_scale': 0.500000},
    'pink1_parkin_OE_age25':{'motor_scale': 0.782609, 'coupling_scale': 0.500000},
    'parkin':               {'motor_scale': 0.985149, 'coupling_scale': 0.476190},
    'dj1':                  {'motor_scale': 0.960396, 'coupling_scale': 0.714286},
    'lrrk2':                {'motor_scale': 1.143564, 'coupling_scale': 0.904762},
    'complexI':             {'motor_scale': 1.108911, 'coupling_scale': 0.857143},
}
print(f"{'model':24} {'motor(new/ref)':>18} {'coupling(new/ref)':>20}  direction")
for model, ref in REFERENCE.items():
    s = results[model]
    dmotor = 'OK' if abs(s['motor_scale'] - ref['motor_scale']) <= 0.05 else 'DIFF'
    dcoup  = 'OK' if abs(s['coupling_scale'] - ref['coupling_scale']) <= 0.05 else 'DIFF'
    print(f"{model:24} {s['motor_scale']:8.4f}/{ref['motor_scale']:<8.4f}"
          f" {s['coupling_scale']:9.4f}/{ref['coupling_scale']:<9.4f}  [{dmotor},{dcoup}]")
print('\nGhi chu: giao thuoc tai notebook la tai lap minh bach tu public API,')
print('khong phai script goc — ky vong khop HUONG, dung lai khop tuyet doi.')

In [ ]:
# @title 12. Tải toàn bộ bridge_scales.json về máy local
try:
    from google.colab import files
    for model in MODELS:
        files.download(str(out_dir / f'{model}_bridge_scales.json'))
except ModuleNotFoundError:
    print('Khong phai Colab — copy thu cong tu:', out_dir)

## Bước tiếp theo (trên máy local)

```powershell
# 1. Copy 7 file *_bridge_scales.json vào drosophila-pd-flygym\data\bridge_scales\
# 2. Chạy body-side (đã xác minh hoạt động):
.venv\Scripts\python.exe scripts\run_brain_driven_experiment.py `
  --scales-json data\bridge_scales\pink1_bridge_scales.json `
  --model-name pink1 `
  --output results\brain_driven\rerun_<ngay>\pink1_locomotion.json
```

Sau đó so với dataset bằng `scripts/compare_brain_driven_rerun.py`.

**Phạm vi khoa học:** kết quả là hành vi mô phỏng tính toán; không xác nhận
bệnh Parkinson sinh học, không phải bằng chứng từ ruồi thật.